# Continuous Batching - Complete Implementation Guide

This notebook covers:
- **Basic Implementation**: Core concepts
- **Advanced Implementation**: Production patterns
- **Real-World Examples**: Industry implementations

Source: `llm/concepts/continuous-batching.md`

## Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time

print("Libraries loaded")

## Basic Implementation

In [ ]:
# Basic Continuous Batching Simulator
import torch
import time
from queue import Queue
from threading import Thread

class ContinuousBatchingServer:
    """Simulate continuous batching for inference"""

    def __init__(self, batch_size=32, wait_time_ms=10):
        self.batch_size = batch_size
        self.wait_time = wait_time_ms / 1000.0
        self.request_queue = Queue()
        self.results = {}

    def add_request(self, request_id, input_ids):
        """Add inference request to queue"""
        self.request_queue.put((request_id, input_ids))

    def process_batch(self):
        """Process requests as they arrive"""
        batch = []
        batch_ids = []

        start_time = time.time()
        while len(batch) < self.batch_size:
            try:
                req_id, input_ids = self.request_queue.get(timeout=self.wait_time)
                batch.append(input_ids)
                batch_ids.append(req_id)
            except:
                # Timeout: process partial batch
                break

        if batch:
            # Process batch
            batch_tensor = torch.stack(batch)
            outputs = self._infer(batch_tensor)

            # Return results
            for req_id, output in zip(batch_ids, outputs):
                self.results[req_id] = output

        return len(batch_ids)

    def _infer(self, batch):
        """Dummy inference"""
        return torch.randn(batch.shape[0], 100)

# Usage
server = ContinuousBatchingServer(batch_size=4)
for i in range(10):
    server.add_request(i, torch.randn(10))

processed = server.process_batch()
print(f"Processed {processed} requests in one batch")

## Advanced Implementation

In [ ]:
# Advanced Continuous Batching with Multiple Batch Sizes
import torch
import time
from typing import Dict, List
from collections import defaultdict

class DynamicBatchingServer:
    """Continuous batching that adapts batch size"""

    def __init__(self, max_batch_size=32, max_wait_ms=50):
        self.max_batch_size = max_batch_size
        self.max_wait = max_wait_ms / 1000.0
        self.requests = []

    def add_request(self, request_id, tokens, priority=1):
        """Add request with priority"""
        self.requests.append({
            'id': request_id,
            'tokens': tokens,
            'priority': priority,
            'added_at': time.time()
        })

    def should_batch(self):
        """Decide whether to process batch"""
        if not self.requests:
            return False
        if len(self.requests) >= self.max_batch_size:
            return True

        # Check timeout
        oldest = min(r['added_at'] for r in self.requests)
        if time.time() - oldest > self.max_wait:
            return True

        return False

    def get_batch(self):
        """Get next batch, respecting priority and max size"""
        if not self.should_batch():
            return None

        # Sort by priority
        self.requests.sort(key=lambda x: -x['priority'])
        batch = self.requests[:self.max_batch_size]
        self.requests = self.requests[self.max_batch_size:]

        tokens = torch.stack([r['tokens'] for r in batch])
        ids = [r['id'] for r in batch]

        return ids, tokens

    def infer_batch(self, tokens):
        """Process batch inference"""
        # Simulate different sequence lengths
        max_len = max(t.shape[0] for t in tokens)

        # Pad sequences
        padded = []
        for t in tokens:
            if t.shape[0] < max_len:
                pad = torch.zeros(max_len - t.shape[0], *t.shape[1:])
                t = torch.cat([t, pad])
            padded.append(t)

        batch = torch.stack(padded)
        outputs = torch.randn(batch.shape[0], 768)
        return outputs

# Usage
server = DynamicBatchingServer(max_batch_size=8, max_wait_ms=100)
for i in range(10):
    tokens = torch.randn(torch.randint(5, 20, (1,)).item())
    server.add_request(i, tokens, priority=torch.rand(1).item())

batch_ids, batch = server.get_batch()
if batch is not None:
    outputs = server.infer_batch(batch)
    print(f"Processed batch of {len(batch_ids)} with shape {outputs.shape}")

## Real-World: Tensorrt

In [ ]:
# Real-World: TensorRT Dynamic Batching
class TensorRTDynamicBatcher:
    """TensorRT's dynamic batching for GPU inference"""

    def __init__(self, max_batch_size=32, queue_delay_ms=1):
        self.max_batch_size = max_batch_size
        self.queue_delay = queue_delay_ms / 1000.0
        self.queue = []
        self.batch_size_distribution = []

    def add_request(self, req_id, input_data):
        """Add request to inference queue"""
        self.queue.append({
            'id': req_id,
            'data': input_data,
            'timestamp': time.time()
        })

    def should_execute_batch(self):
        """Decide if batch should execute"""
        if len(self.queue) == 0:
            return False
        if len(self.queue) >= self.max_batch_size:
            return True

        # Time-out based batching
        oldest_request = min(r['timestamp'] for r in self.queue)
        age = time.time() - oldest_request
        if age > self.queue_delay:
            return True

        return False

    def execute_batch(self):
        """Execute current batch"""
        if not self.should_execute_batch():
            return None

        batch_size = min(len(self.queue), self.max_batch_size)
        batch = self.queue[:batch_size]
        self.queue = self.queue[batch_size:]

        # Record batch size distribution for analysis
        self.batch_size_distribution.append(batch_size)

        # Execute on GPU
        return batch_size

    def get_stats(self):
        """Get batching statistics"""
        if not self.batch_size_distribution:
            return {}

        sizes = self.batch_size_distribution
        return {
            'avg_batch_size': sum(sizes) / len(sizes),
            'max_batch_size': max(sizes),
            'min_batch_size': min(sizes),
            'total_batches': len(sizes)
        }

# Usage
import time
batcher = TensorRTDynamicBatcher(max_batch_size=16)
for i in range(50):
    batcher.add_request(i, torch.randn(1, 10))

while True:
    size = batcher.execute_batch()
    if size is None:
        break

stats = batcher.get_stats()
print(f"Batching stats: {stats}")

## Real-World: Vllm

In [ ]:
# Real-World: vLLM Continuous Batching
# vLLM implements production continuous batching
from typing import List, Tuple
import torch

class vLLMStyleBatcher:
    """Inspired by vLLM's continuous batching scheduler"""

    def __init__(self, max_batch_size=32):
        self.max_batch_size = max_batch_size
        self.waiting_queue = []
        self.running_batch = {}

    def schedule(self, request_id, prompt_len, generation_len):
        """Schedule request: track prompt + generation tokens"""
        self.waiting_queue.append({
            'id': request_id,
            'prompt_len': prompt_len,
            'gen_len': generation_len,
            'tokens_generated': 0
        })

    def get_batch_for_step(self) -> Tuple[List, int]:
        """Get batch for next generation step"""
        # Prioritize by: generation progress, then by age
        self.waiting_queue.sort(
            key=lambda x: (x['tokens_generated'], -x['id'])
        )

        batch = self.waiting_queue[:self.max_batch_size]
        batch_ids = [r['id'] for r in batch]

        return batch_ids, len(batch)

    def advance_generation(self):
        """Advance all requests in batch by 1 token"""
        for req in self.waiting_queue:
            req['tokens_generated'] += 1

            # Remove finished requests
            if req['tokens_generated'] >= req['gen_len']:
                self.waiting_queue.remove(req)

# Usage: continuous batching in LLM serving
batcher = vLLMStyleBatcher(max_batch_size=16)
for i in range(100):
    batcher.schedule(i, prompt_len=50, generation_len=100)

# Each step of generation
for step in range(100):
    batch_ids, size = batcher.get_batch_for_step()
    if not batch_ids:
        break

    # Infer on batch
    batcher.advance_generation()

    if step < 3:
        print(f"Step {step}: batch size {size}")

## Resources

- **Markdown**: `llm/concepts/continuous-batching.md`
- **Interview Q&A**: See markdown file
- **Real-world**: Review code above
- **Next Steps**: Try modifying the code